# Data Preparation: Alamedin & Ala-Archa Basins

Preprocessing of meteorological forcing, discharge observations, and catchment
descriptors for two glaciated basins in the Tien Shan (Kyrgyzstan). Produces
`Forcing`, `StreamflowSeries`, and `Catchment` objects ready for
GR6J+CemaNeige+Glacier calibration.

In [1]:
from pathlib import Path
from typing import NamedTuple

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.features
import seaborn as sns

from wat_mod_giz import Catchment, Forcing, StreamflowSeries


class BasinConfig(NamedTuple):
    name: str
    folder: str
    area_km2: float
    rgi_file: str
    gauge_code: int
    gauge_lat: float


BASINS = {
    "15189": BasinConfig(
        name="Alamedin",
        folder="15189_Alamedin_River_KGZ",
        area_km2=414.05,
        rgi_file="RGI_Alamedin.shp",
        gauge_code=15189,
        gauge_lat=42.697,
    ),
    "15194": BasinConfig(
        name="Ala-Archa",
        folder="15194_AlaArcha_River_KGZ",
        area_km2=270.84,
        rgi_file="RGI_Ala-Archa.shp",
        gauge_code=15194,
        gauge_lat=42.660,
    ),
}

DATA_ROOT = Path.home() / "Desktop" / "2025_02_TW_ETHZ_CA_IWRM" / "02_data"
GIS_ROOT = DATA_ROOT / "15189_Alamedin_River_KGZ" / "00_gis"
N_LAYERS = 5
WARMUP = 365

## Potential Evapotranspiration — Oudin (2005)

A temperature-only PET estimate requiring just mean daily temperature $T$ in $^\circ\mathrm{C}$, day of year $J$, and station latitude $\phi$ in radians.

### Extraterrestrial radiation $R_a$

| Quantity | Formula |
|---|---|
| Solar declination | $\delta = 0.4093 \sin\left(\frac{2\pi J}{365} - 1.405\right)$ |
| Sunset hour angle | $\omega_s = \arccos(-\tan\phi\,\tan\delta)$ |
| Inverse relative distance | $d_r = 1 + 0.033\cos\left(\frac{2\pi J}{365}\right)$ |

$$
R_a = \frac{24 \times 60}{\pi}\, G_{sc}\, d_r
      \left[\omega_s \sin\phi\,\sin\delta
            + \cos\phi\,\cos\delta\,\sin\omega_s\right]
$$

where $G_{sc} = 0.0820\,\mathrm{MJ\,m^{-2}\,min^{-1}}$.

### PET

$$
\operatorname{PET} =
\begin{cases}
\frac{R_a}{\lambda\,\rho}\,\frac{T + 5}{100}
  & \text{if } T + 5 > 0 \\[6pt]
0 & \text{otherwise}
\end{cases}
$$

with $\lambda = 2.45\,\mathrm{MJ\,kg^{-1}}$ (latent heat of vaporisation) and $\rho = 1000\,\mathrm{kg\,m^{-3}}$ (density of water).

In [2]:
GSC = 0.0820  # solar constant [MJ m⁻² min⁻¹]
LAMBDA = 2.45  # latent heat of vaporisation [MJ kg⁻¹]
RHO = 1000.0  # density of water [kg m⁻³]


def extraterrestrial_radiation(doy: np.ndarray, lat_rad: float) -> np.ndarray:
    """Daily extraterrestrial radiation R_a [MJ m⁻² day⁻¹] (FAO-56)."""
    dr = 1.0 + 0.033 * np.cos(2.0 * np.pi / 365.0 * doy)
    delta = 0.4093 * np.sin(2.0 * np.pi / 365.0 * doy - 1.405)
    ws = np.arccos(-np.tan(lat_rad) * np.tan(delta))
    return (
        (24.0 * 60.0 / np.pi)
        * GSC
        * dr
        * (ws * np.sin(lat_rad) * np.sin(delta) + np.cos(lat_rad) * np.cos(delta) * np.sin(ws))
    )


def oudin_pet(temp: np.ndarray, doy: np.ndarray, latitude_deg: float) -> np.ndarray:
    """Oudin (2005) PET [mm day⁻¹] from mean daily temperature."""
    lat_rad = np.radians(latitude_deg)
    ra = extraterrestrial_radiation(doy, lat_rad)
    return np.where(temp + 5.0 > 0.0, ra / (LAMBDA * RHO) * (temp + 5.0) / 100.0, 0.0)

## Solid Precipitation Partitioning

CemaNeige requires the mean annual solid precipitation $P_s$ as a catchment descriptor. Solid fraction is estimated with the USACE linear rain/snow threshold:

$$
f_s =
\begin{cases}
1, & T \leq T_s \\
\frac{T_r - T}{T_r - T_s}, & T_s < T < T_r \\
0, & T \geq T_r
\end{cases}
$$

with $T_s = -1^\circ\mathrm{C}$ and $T_r = 3^\circ\mathrm{C}$.

Mean annual solid precipitation ($\mathrm{mm\,yr^{-1}}$):

$$
P_s = 365.25 \times \frac{1}{N}\sum_{i=1}^{N} f_{s,i} P_i
$$

In [3]:
T_SNOW = -1.0  # all-snow threshold [°C]
T_RAIN = 3.0  # all-rain threshold [°C]


def solid_fraction(temp: np.ndarray) -> np.ndarray:
    """USACE linear rain/snow partitioning."""
    return np.clip((T_RAIN - temp) / (T_RAIN - T_SNOW), 0.0, 1.0)


def mean_annual_solid_precip(precip: np.ndarray, temp: np.ndarray) -> float:
    """Mean annual solid precipitation [mm yr⁻¹]."""
    daily_solid = precip * solid_fraction(temp)
    return float(np.mean(daily_solid) * 365.25)

## Load & Process Forcing Data

**Data sources:** ERA5-Land daily basin-averaged precipitation ($\mathrm{mm\,day^{-1}}$) and temperature ($^\circ\mathrm{C}$), stored as CSVs with a `date` index column.

**Discharge conversion** from volumetric to specific:

$$
Q_{\mathrm{mm\,day^{-1}}} = Q_{\mathrm{m^3\,s^{-1}}} \times \frac{86.4}{A_{\mathrm{km^2}}}
$$

(factor $86.4 = 86\,400\,\mathrm{s\,day^{-1}} / 1\,000\,\mathrm{mm\,m^{-1}} \times 1\,\mathrm{km^2} / 10^6\,\mathrm{m^2}$
simplifies to $86\,400 / 10^6 \times 10^3 = 0.0864 \times 10^3$).

**Strategy:**
1. Find the longest contiguous (no-gap) period with both forcing and discharge.
2. Extend forcing **backward** by the warmup period so the model state can equilibrate before the evaluation window begins.

In [ ]:
def longest_contiguous_run(dates: pd.DatetimeIndex) -> slice:
    """Return a slice for the longest run of consecutive daily dates."""
    gaps = dates.to_series().diff().dt.days.ne(1)
    group_ids = gaps.cumsum()
    group_sizes = group_ids.value_counts()
    best_group = group_sizes.idxmax()
    mask = group_ids == best_group
    idx = mask.values.nonzero()[0]
    return slice(idx[0], idx[-1] + 1)


basin_data: dict[str, dict] = {}

for code, cfg in BASINS.items():
    base = DATA_ROOT / cfg.folder

    # -- ERA5 forcing --------------------------------------------------------
    precip_df = pd.read_csv(base / "02_forcing" / "era5_precipitation_daily.csv", parse_dates=["date"])
    temp_df = pd.read_csv(base / "02_forcing" / "era5_temperature_daily.csv", parse_dates=["date"])
    era5 = precip_df.merge(temp_df, on="date")

    precip_arr = era5["precipitation_mm"].values
    temp_arr = era5["temperature_degC"].values
    doy_arr = era5["date"].dt.dayofyear.values 
    era5["pet_mm"] = oudin_pet(era5["temperature_degC"].values, era5["date"].dt.dayofyear.values, cfg.gauge_lat)

    # -- Mean annual solid precip (full ERA5 record) -------------------------
    masp = mean_annual_solid_precip(precip_arr, temp_arr)

    # -- Observed discharge --------------------------------------------------
    q_df = pd.read_csv(base / "01_gauge_data" / f"q{code}_not_to_be_shared.csv")
    q_df["date"] = pd.to_datetime(q_df["local_date_time"]).dt.normalize()
    q_df["q_mm"] = q_df["q_m3s"] * 86.4 / cfg.area_km2

    # -- Overlap & longest contiguous run ------------------------------------
    era5_dates = pd.DatetimeIndex(era5["date"])
    q_dates = pd.DatetimeIndex(q_df["date"])
    overlap = era5_dates.intersection(q_dates)
    run = longest_contiguous_run(overlap.sort_values())
    obs_dates = overlap.sort_values()[run]

    obs_start, obs_end = obs_dates[0], obs_dates[-1]
    forcing_start = obs_start - pd.Timedelta(days=WARMUP)

    # -- Slice forcing (warmup + observation window) -------------------------
    forcing_mask = (era5["date"] >= forcing_start) & (era5["date"] <= obs_end)
    f = era5.loc[forcing_mask].reset_index(drop=True)

    forcing = Forcing(
        time=f["date"].values.astype("datetime64[D]"),
        precip=f["precipitation_mm"].values,
        pet=f["pet_mm"].values,
        temp=f["temperature_degC"].values,
    )

    # -- Slice observed (must match forcing.time[WARMUP:]) -------------------
    obs_mask = q_df["date"].isin(obs_dates)
    q_sub = q_df.loc[obs_mask].sort_values("date").reset_index(drop=True)

    observed = StreamflowSeries(
        time=forcing.time[WARMUP:],
        streamflow=q_sub["q_mm"].values,
    )

    basin_data[code] = {
        "forcing": forcing,
        "observed": observed,
        "masp": masp,
        "cfg": cfg,
    }

    print(
        f"{cfg.name}: forcing {forcing.time[0]}–{forcing.time[-1]} ({len(forcing)} days), "
        f"observed {obs_start.date()}–{obs_end.date()} ({len(observed)} days), "
        f"mean annual solid precip = {masp:.0f} mm/yr"
    )

Alamedin: forcing 2018-09-30T00:00:00.000000000–2022-01-21T00:00:00.000000000 (1210 days), observed 2019-09-30–2022-01-21 (845 days), mean annual solid precip = 410 mm/yr
Ala-Archa: forcing 2018-09-29T00:00:00.000000000–2023-05-17T00:00:00.000000000 (1692 days), observed 2019-09-29–2023-05-17 (1327 days), mean annual solid precip = 429 mm/yr


## Hypsometric Curve

A 101-element array giving the elevation (m a.s.l.) below which each percentile
(0 %, 1 %, ..., 100 %) of the catchment area lies. CemaNeige uses the
hypsometric curve to partition the catchment into $N$ elevation bands with
equal area, each receiving its own snow and glacier accounting.

**Computation:** extract all DEM pixel values inside the catchment boundary,
then take `np.percentile(elevations, range(101))`.

In [5]:
hypsometric_curves: dict[str, np.ndarray] = {}

for code, cfg in BASINS.items():
    dem_path = DATA_ROOT / cfg.folder / "02_forcing" / "dem.tif"
    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        valid = dem[~np.isnan(dem)]

    curve = np.percentile(valid, np.arange(101))
    hypsometric_curves[code] = curve
    print(f"{cfg.name}: elevation {curve[0]:.0f}–{curve[-1]:.0f} m a.s.l. (median {curve[50]:.0f} m)")

Alamedin: elevation 1329–4818 m a.s.l. (median 3301 m)
Ala-Archa: elevation 1539–4834 m a.s.l. (median 3336 m)


In [ ]:
sns.set_context("paper", fontscale=1.3)

fig, ax = plt.subplots(figsize=(6, 6))
percentiles = np.arange(101)

for code, cfg in BASINS.items():
    sns.lineplot(x=percentiles, y=hypsometric_curves[code], ax=ax, label=cfg.name)

ax.set_xlabel("Catchment area percentile [%]")
ax.set_ylabel("Elevation [m a.s.l.]")
ax.set_title("Hypsometric curves")
sns.despine()
plt.show()


## Gauge Elevation

`input_elevation` is the elevation (m a.s.l.) of the meteorological reference
point — here the gauging station. It is sampled from the DEM at the gauge
coordinates and serves as the anchor for temperature and precipitation lapse-rate
extrapolation across elevation bands.

In [6]:
gauges = gpd.read_file(GIS_ROOT / "Ala-Archa_Alamedin_gauges.shp")
gauge_elevations: dict[str, float] = {}

for code, cfg in BASINS.items():
    gauge = gauges.loc[gauges["CODE"] == str(cfg.gauge_code)].iloc[0]
    lon, lat = gauge.geometry.x, gauge.geometry.y

    dem_path = DATA_ROOT / cfg.folder / "02_forcing" / "dem.tif"
    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        row, col = src.index(lon, lat)
        elev = dem[row, col]

        # Gauge at outlet may fall outside valid DEM — use nearest valid pixel
        if np.isnan(elev):
            valid_mask = ~np.isnan(dem)
            rows, cols = np.where(valid_mask)
            dists = (rows - row) ** 2 + (cols - col) ** 2
            nearest = dists.argmin()
            elev = dem[rows[nearest], cols[nearest]]

    gauge_elevations[code] = float(elev)
    print(f"{cfg.name} gauge: ({lon:.3f}°E, {lat:.3f}°N) → {elev:.0f} m a.s.l.")

Alamedin gauge: (74.664°E, 42.697°N) → 1330 m a.s.l.
Ala-Archa gauge: (74.501°E, 42.660°N) → 1539 m a.s.l.


## Glacier Fractions per Elevation Band

Each of the $N$ elevation bands requires a glacier coverage fraction
$f_{\text{glacier}}^{(k)} \in [0, 1]$.

**Steps:**
1. Rasterize the RGI glacier outlines onto the DEM grid (binary mask).
2. Derive elevation-band boundaries from the hypsometric curve
   (equal-area quantiles).
3. For each band $k$, compute

$$
f_{\text{glacier}}^{(k)}
  = \frac{n_{\text{glacier}}^{(k)}}{n_{\text{total}}^{(k)}}
$$

where $n_{\text{glacier}}^{(k)}$ is the number of glacierised DEM pixels in
band $k$ and $n_{\text{total}}^{(k)}$ is the total pixel count in that band.

In [7]:
glacier_fractions: dict[str, np.ndarray] = {}

for code, cfg in BASINS.items():
    rgi = gpd.read_file(GIS_ROOT / cfg.rgi_file)

    dem_path = DATA_ROOT / cfg.folder / "02_forcing" / "dem.tif"
    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        transform = src.transform
        shape = src.shape

    glacier_mask = rasterio.features.rasterize(
        [(geom, 1) for geom in rgi.geometry],
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype=np.uint8,
    )

    valid = ~np.isnan(dem)

    # Elevation band boundaries: equal-area percentiles over valid pixels
    pct_bounds = np.linspace(0, 100, N_LAYERS + 1)
    elev_bounds = np.percentile(dem[valid], pct_bounds)

    fracs = np.zeros(N_LAYERS, dtype=np.float64)
    for k in range(N_LAYERS):
        lo, hi = elev_bounds[k], elev_bounds[k + 1]
        band = valid & (dem >= lo) & (dem < hi) if k < N_LAYERS - 1 else valid & (dem >= lo) & (dem <= hi)

        n_total = band.sum()
        n_glacier = (band & (glacier_mask == 1)).sum()
        fracs[k] = n_glacier / n_total if n_total > 0 else 0.0

    glacier_fractions[code] = fracs
    print(f"{cfg.name} glacier fractions: {[f'{f:.3f}' for f in fracs]}")

Alamedin glacier fractions: ['0.000', '0.000', '0.025', '0.241', '0.316']
Ala-Archa glacier fractions: ['0.000', '0.000', '0.020', '0.204', '0.375']


## Assemble `Catchment` Objects

All components are now ready. Each basin gets a `Catchment` instance:

```python
Catchment(
    mean_annual_solid_precip,  # mm/yr — from solid-precip partitioning
    n_layers,                  # number of elevation bands (N_LAYERS)
    hypsometric_curve,         # 101-element elevation percentile array
    input_elevation,           # gauge elevation from DEM
    glacier_fractions,         # per-band glacier coverage [0, 1]
)
```

Together with `Forcing` and `StreamflowSeries` built earlier, this is
everything the calibration step needs.

In [8]:
catchments: dict[str, Catchment] = {}

for code, cfg in BASINS.items():
    catchment = Catchment(
        mean_annual_solid_precip=basin_data[code]["masp"],
        n_layers=N_LAYERS,
        hypsometric_curve=hypsometric_curves[code],
        input_elevation=gauge_elevations[code],
        glacier_fractions=glacier_fractions[code],
    )
    catchments[code] = catchment

    curve = hypsometric_curves[code]
    fracs = glacier_fractions[code]
    print(
        f"\n{cfg.name}:\n"
        f"  mean annual solid precip = {basin_data[code]['masp']:.0f} mm/yr\n"
        f"  n_layers = {N_LAYERS}\n"
        f"  elevation range = {curve[0]:.0f}–{curve[-1]:.0f} m\n"
        f"  gauge elevation = {gauge_elevations[code]:.0f} m\n"
        f"  max glacier fraction = {fracs.max():.3f} (band {fracs.argmax()})"
    )


Alamedin:
  mean annual solid precip = 410 mm/yr
  n_layers = 5
  elevation range = 1329–4818 m
  gauge elevation = 1330 m
  max glacier fraction = 0.316 (band 4)

Ala-Archa:
  mean annual solid precip = 429 mm/yr
  n_layers = 5
  elevation range = 1539–4834 m
  gauge elevation = 1539 m
  max glacier fraction = 0.375 (band 4)
